In [ ]:
# example_decompose2d_synthetic_kelvin.py
# Generate a synthetic 2D field with several wave packets + a Kelvin wake,
# then run juwavelet's 2D CWT and plot components. No files written.

from __future__ import annotations
import math
import string
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Sequence, Tuple

# Inline plotting if you're in Jupyter
try:
    get_ipython  # type: ignore[name-defined]
    %matplotlib inline
except Exception:
    pass

from juwavelet import transform, utils

# --------------------------
# Plot style
# --------------------------
matplotlib.rcParams.update({
    'axes.labelsize': 16,
    'font.size': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.figsize': [2 * 6.94, 2 * 4.29],
})

# --------------------------
# Wave packets
# --------------------------
@dataclass
class WavePacket:
    """2D plane-wave packet with Gaussian envelope.
       A * exp(-((x-x0)^2/(2*sx^2) + (y-y0)^2/(2*sy^2))) * cos(kx*x + ky*y + phi)
       Provide (kx, ky) or (k, theta) in radians.
    """
    amplitude: float = 1.0
    kx: float | None = None
    ky: float | None = None
    k: float | None = None
    theta: float | None = None
    x0: float = 0.0
    y0: float = 0.0
    sx: float = 10.0
    sy: float = 10.0
    phi: float = 0.0

    def k_components(self) -> Tuple[float, float]:
        if self.kx is not None and self.ky is not None:
            return self.kx, self.ky
        if self.k is None or self.theta is None:
            raise ValueError("Provide either (kx, ky) or (k, theta) for WavePacket.")
        return self.k * math.cos(self.theta), self.k * math.sin(self.theta)

# --------------------------
# Grid + helpers
# --------------------------
def make_grid(nx: int, ny: int, dx: float, dy: float) -> Tuple[np.ndarray, np.ndarray]:
    x = np.arange(nx, dtype=float) * dx
    y = np.arange(ny, dtype=float) * dy
    X, Y = np.meshgrid(x, y)  # shapes (ny, nx)
    return X, Y

def packet_field(xs: np.ndarray, ys: np.ndarray, pkt: WavePacket) -> np.ndarray:
    kx, ky = pkt.k_components()
    gauss = np.exp(-(((xs - pkt.x0) ** 2) / (2 * pkt.sx ** 2) + ((ys - pkt.y0) ** 2) / (2 * pkt.sy ** 2)))
    phase = kx * xs + ky * ys + pkt.phi
    return pkt.amplitude * gauss * np.cos(phase)

# --------------------------
# Kelvin wake (simplified)
# --------------------------
def kelvin_wake_field(
    xs: np.ndarray,
    ys: np.ndarray,
    *,
    x0: float,
    y0: float,
    k0: float = 2 * math.pi / 40.0,  # dominant wavenumber
    amp: float = 2.0,
    nt: int = 181,                   # number of direction samples inside wedge
    theta_wedge_deg: float = 19.47,  # Kelvin wedge half angle ~ arcsin(1/3)
    envelope_sigma: float = 120.0,   # Gaussian decay around source
    radial_damp: float = 0.35,       # 1/sqrt(r) style damping factor
) -> np.ndarray:
    """
    Build a simple Kelvin ship-wake pattern by summing plane waves inside the
    Kelvin wedge [-theta_K, +theta_K]. This is not a full dispersion solution,
    but it produces the classic V-shaped interference.
    Track direction = +x.

    Parameters
    ----------
    k0 : base wavenumber
    amp : overall amplitude
    nt  : number of angular samples in wedge
    theta_wedge_deg : Kelvin wedge half-angle
    envelope_sigma : Gaussian envelope around source to localize the wake
    radial_damp : strength of radial damping 1/sqrt(r)

    Returns
    -------
    ndarray field with shape like xs/ys (ny, nx)
    """
    # Shift coords relative to source
    Xc = xs - x0
    Yc = ys - y0
    r = np.hypot(Xc, Yc)
    # Avoid division by zero at the source
    r_safe = np.maximum(r, 1.0)

    # Angular samples in wedge
    thK = math.radians(theta_wedge_deg)
    thetas = np.linspace(-thK, thK, nt)

    field = np.zeros_like(xs, dtype=float)

    # Angular weighting to emphasize wedge edges a bit
    # and include transverse waves near 0 deg
    theta_weight = 0.6 + 0.4 * (np.cos(thetas / thK * math.pi / 2) ** 2)

    for w, theta in zip(theta_weight, thetas):
        # Directional plane wave along angle theta relative to +x
        phase = k0 * (Xc * math.cos(theta) + Yc * math.sin(theta))
        field += w * np.cos(phase)

    # Normalize by number of angles
    field /= nt

    # Apply distance damping and a soft Gaussian around the source
    field *= amp * (1.0 / np.sqrt(1.0 + radial_damp * r_safe)) * np.exp(-(r**2) / (2 * envelope_sigma**2))

    return field

# --------------------------
# Synthetic field with Kelvin wake
# --------------------------
def generate_synthetic_field_with_kelvin(
    nx: int = 256,
    ny: int = 256,
    dx: float = 2.0,
    dy: float = 2.0,
    packets: Sequence[WavePacket] | None = None,
    include_kelvin: bool = True,
    kelvin_params: dict | None = None,
    noise_sigma: float = 0.10,
    seed: int | None = 0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns xs, ys, total_field, kelvin_only with shapes (ny, nx).
    """
    rng = np.random.default_rng(seed) if seed is not None else np.random.default_rng()
    xs, ys = make_grid(nx, ny, dx, dy)
    Lx = nx * dx
    Ly = ny * dy

    if packets is None:
        packets = [
            WavePacket(amplitude=3.0, k=2 * math.pi / 120.0, theta=math.radians(45),
                       x0=0.35 * Lx, y0=0.30 * Ly, sx=86, sy=56, phi=0.0),
            WavePacket(amplitude=2.5, k=2 * math.pi / 16.0, theta=math.radians(28),
                       x0=0.60 * Lx, y0=0.55 * Ly, sx=38, sy=28, phi=0.6),
        ]

    field = np.zeros_like(xs)
    for pkt in packets:
        field += packet_field(xs, ys, pkt)

    kelvin_only = np.zeros_like(xs)
    if include_kelvin:
        kp = dict(
            x0=0.25 * Lx,
            y0=0.70 * Ly,
            k0=2 * math.pi / 40.0,
            amp=8.2,
            nt=181,
            theta_wedge_deg=19.47,
            envelope_sigma=140.0,
            radial_damp=0.30,
        )
        if kelvin_params:
            kp.update(kelvin_params)
        kelvin_only = kelvin_wake_field(xs, ys, **kp)
        field += kelvin_only

    if noise_sigma > 0:
        field = field + rng.normal(0.0, noise_sigma, size=field.shape)

    return xs, ys, field, kelvin_only

# --------------------------
# Quicklook
# --------------------------
def quicklook(x1d: np.ndarray, y1d: np.ndarray, Z: np.ndarray, title: str) -> None:
    fig, ax = plt.subplots()
    im = ax.pcolormesh(x1d, y1d, Z, shading='auto')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("Amplitude (arb)")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    plt.tight_layout()

# --------------------------
# Pipeline
# --------------------------
def run_pipeline():
    # Build synthetic field with Kelvin wake
    xs2d, ys2d, wave, kelvin_only = generate_synthetic_field_with_kelvin(
        nx=256, ny=256, dx=2.0, dy=2.0, include_kelvin=True, seed=0
    )

    x1d = xs2d[0, :].copy()
    y1d = ys2d[:, 0].copy()
    x1d -= x1d[0]
    dx = np.diff(x1d).mean()
    dy = np.diff(y1d).mean()

    # Quicklooks
    quicklook(x1d, y1d, wave, "Synthetic field: packets + Kelvin wake")
    quicklook(x1d, y1d, kelvin_only, "Kelvin wake component only")

    # juwavelet expects (nx, ny) so transpose
    wave_xy = wave.T

    cwt = transform.decompose2d(
        wave_xy, dx=dx, dy=dy, s0=10, dj=0.25, js=20, jt=36, aspect=1
    )

    # Identify waves
    amps, idxs, iwave = utils.identify_cluster2d(cwt, min_amp=1.0, thr=0.25)

    decomposition, period, theta = [cwt[k] for k in ("decomposition", "period", "theta")]
    orig = decomposition.copy()

    # Overview CWT plot
    cmap = matplotlib.cm.turbo
    norm = matplotlib.colors.BoundaryNorm(
        np.exp(np.linspace(np.log(0.12), np.log(2.5), 10)), cmap.N
    )
    fig1, _ = utils.plot_decomposition2d(cwt, redux_s=2, redux_t=2, cmap=cmap, norm=norm)

    # Component panels (like before)
    fig2, axs = plt.subplots(2, 3, figsize=(12, 10))
    axs = axs.T  # shape (3, 2)
    opts = {"cmap": "RdBu_r", "vmin": -5, "vmax": 5, "rasterized": True}

    # original
    axs[0, 0].set_title("original")
    axs[0, 0].pcolormesh(x1d, y1d, wave, **opts)

    # left slanted (theta > pi/2 masked out)
    decomposition[:] = orig
    decomposition[:, (np.pi / 2 < theta)] = 0
    rec = transform.reconstruct2d(cwt)  # (nx, ny)
    axs[1, 0].set_title("left slanted")
    axs[1, 0].pcolormesh(x1d, y1d, rec.T, **opts)

    # right slanted (theta <= pi/2 masked out)
    decomposition[:] = orig
    decomposition[:, (theta <= np.pi / 2)] = 0
    rec = transform.reconstruct2d(cwt)
    axs[2, 0].set_title("right slanted")
    axs[2, 0].pcolormesh(x1d, y1d, rec.T, **opts)

    # low-pass (period >= 100 kept)
    decomposition[:] = orig
    decomposition[period < 100] = 0
    rec = transform.reconstruct2d(cwt)
    axs[0, 1].set_title("low pass")
    axs[0, 1].pcolormesh(x1d, y1d, rec.T, **opts)

    # highlight two identified waves
    demo_indices = [0, 1]
    targets = [(1, axs[1, 1]), (2, axs[2, 1])]
    for (demo_idx, (rowi, ax)) in zip(demo_indices, targets):
        if isinstance(idxs, (list, tuple)) and demo_idx < len(idxs):
            decomposition[:] = orig
            decomposition[iwave != demo_idx] = 0
            udx = idxs[demo_idx]
            rec = transform.reconstruct2d(cwt)
            ax.set_title(
                rf"$\lambda_x$={cwt['wavelength_x'][udx[0], udx[1]]:3.0f} "
                rf"$\lambda_z$={cwt['wavelength_y'][udx[0], udx[1]]:3.0f}"
            )
            ax.pcolormesh(x1d, y1d, rec.T, **opts)

            # mark peak location
            decomposition[:] = np.abs(decomposition)
            max_idx = np.unravel_index(
                decomposition[udx[0], udx[1]].argmax(), decomposition[0, 0].shape
            )
            ax.plot(x1d[max_idx[0]], y1d[max_idx[1]], "o", color="w", mec="k", ms=10)
            print(f"Wave {demo_idx}: peak amp = {decomposition[udx[0], udx[1]].max():.3f}")
        else:
            ax.set_title(f"wave {demo_idx} not found")
            ax.axis("off")

    # labels
    for ax in axs[:, 1]:
        ax.set_xlabel("distance (km)")
    for ax in axs[0, :]:
        ax.set_ylabel("altitude (km)")

    # letter tags
    letterbox = {"boxstyle": "circle", "lw": 0.67, "facecolor": "white", "edgecolor": "black"}
    for ax, letter in zip(axs.T.reshape(-1), string.ascii_lowercase):
        ax.text(0.12, 0.15, letter, transform=ax.transAxes,
                bbox=letterbox, va="top", ha="right", weight="bold")

    fig2.tight_layout()
    plt.show(fig1)
    plt.show(fig2)

# --------------------------
# Run
# --------------------------
run_pipeline()


In [2]:
# %%
# ============================================================
# Plot phase of reconstructed packets / components
#   - Reconstruct (ideally complex) field after masking decomposition
#   - Plot amplitude and phase side-by-side
#   - Mask phase where amplitude is small (avoids noisy phase)
# Units are km on both axes (since x1d, y1d are in km here)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

def plot_recon_amp_phase(ax_amp, ax_phase, x1d, y1d, rec_xy, amp_floor=0.20, title=""):
    """
    rec_xy: reconstructed field in (nx, ny). Complex preferred.
    amp_floor: mask phase where amplitude < amp_floor * max(amplitude).
    """
    if np.iscomplexobj(rec_xy):
        amp = np.abs(rec_xy)
        phase = np.angle(rec_xy)
    else:
        amp = np.abs(rec_xy)
        phase = np.full_like(amp, np.nan, dtype=float)

    thr = amp_floor * float(np.nanmax(amp)) if np.nanmax(amp) > 0 else 0.0
    phase_masked = np.where(amp >= thr, phase, np.nan)

    # Amplitude
    im1 = ax_amp.pcolormesh(x1d, y1d, amp.T, shading="auto")
    ax_amp.set_title(f"{title} amplitude")
    ax_amp.set_xlabel("x (km)")
    ax_amp.set_ylabel("y (km)")
    plt.colorbar(im1, ax=ax_amp, fraction=0.046, pad=0.04)

    # Phase
    im2 = ax_phase.pcolormesh(
        x1d, y1d, phase_masked.T, shading="auto", vmin=-np.pi, vmax=np.pi
    )
    ax_phase.set_title(f"{title} phase")
    ax_phase.set_xlabel("x (km)")
    ax_phase.set_ylabel("y (km)")
    cb = plt.colorbar(im2, ax=ax_phase, fraction=0.046, pad=0.04)
    cb.set_label("phase (rad)")

def recon_component(cwt, orig_decomp, mask, label, x1d, y1d, amp_floor=0.20):
    """
    Apply a boolean mask to theta/period etc. by zeroing decomposition entries,
    reconstruct, then plot amplitude + phase.
    mask should be broadcast-compatible with cwt["decomposition"].
    """
    decomp = cwt["decomposition"]
    decomp[:] = orig_decomp
    decomp[mask] = 0

    rec_xy = transform.reconstruct2d(cwt)  # (nx, ny), complex if reconstruct2d preserves it

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    plot_recon_amp_phase(ax1, ax2, x1d, y1d, rec_xy, amp_floor=amp_floor, title=label)
    fig.tight_layout()

    print(f"{label}: reconstruct2d complex? {np.iscomplexobj(rec_xy)}  dtype={rec_xy.dtype}")
    return rec_xy

# -----------------------------
# Use after you already built cwt in your pipeline
# Assumes you have:
#   cwt, x1d, y1d
#   orig = cwt["decomposition"].copy()
#   theta = cwt["theta"]
#   period = cwt["period"]
# -----------------------------

# Keep a copy of the original decomposition (complex)
orig = cwt["decomposition"].copy()
theta = cwt["theta"]
period = cwt["period"]

# Example 1: left slanted (mask out theta > pi/2)
_ = recon_component(
    cwt=cwt,
    orig_decomp=orig,
    mask=(np.pi / 2 < theta),
    label="left slanted",
    x1d=x1d, y1d=y1d,
    amp_floor=0.20
)

# Example 2: right slanted (mask out theta <= pi/2)
_ = recon_component(
    cwt=cwt,
    orig_decomp=orig,
    mask=(theta <= np.pi / 2),
    label="right slanted",
    x1d=x1d, y1d=y1d,
    amp_floor=0.20
)

# Example 3: low-pass (mask out period < 100 km)
_ = recon_component(
    cwt=cwt,
    orig_decomp=orig,
    mask=(period < 100),
    label="low pass (period ≥ 100 km)",
    x1d=x1d, y1d=y1d,
    amp_floor=0.20
)

# Example 4: plot phase for identified packet indices (0 and 1) if available
# This uses your iwave labeling and zeros everything not equal to the target wave index.
if "iwave" in locals() and iwave is not None:
    for demo_idx in [0, 1]:
        _ = recon_component(
            cwt=cwt,
            orig_decomp=orig,
            mask=(iwave != demo_idx),
            label=f"wave {demo_idx}",
            x1d=x1d, y1d=y1d,
            amp_floor=0.25
        )

# Restore decomposition when done
cwt["decomposition"][:] = orig


NameError: name 'cwt' is not defined